# FinChart-R2 — Phase 2C Teacher Raw Analysis and DPO Export

This local notebook performs no teacher calls. It converts raw teacher messages into reproducible analysis routes, applying deterministic checks before any provisional DPO pair is exported.


## 1. Configure local input and outputs

This notebook reads the raw-capture artifact from Notebook 06. It never reads `.env` and never exposes credentials.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r'D:\Finance_AI\FinChart-R2')
RESULTS_DIR = PROJECT_ROOT / 'results' / 'finchart_r2_phase2c_train_mining'
RAW_INPUT = RESULTS_DIR / 'dpo_train_teacher_raw_capture.jsonl'
OUTPUT_PREFIX = 'phase2c_teacher_v1'

assert RAW_INPUT.is_file(), f'Run Notebook 06 raw capture first: {RAW_INPUT}'
print('Input:', RAW_INPUT)
print('Output directory:', RESULTS_DIR)


## 2. Normalize and deterministically route teacher records

`DPO_CANDIDATE_PROVISIONAL` requires a normalized `MODEL_ERROR` verdict plus both the teacher answer and the final answer in its corrected response matching ChartQA ground truth. These candidates still require manual audit before training.


In [ ]:
import json
import re
import string
from collections import Counter

ANSWER_LINE = re.compile(r'^\s*(?:final\s+answer|answer)\s*:\s*(.+?)\s*$', re.I)
VERDICT_ALIASES = {
    'MODEL_ERROR': 'MODEL_ERROR',
    'MODEL_EROR': 'MODEL_ERROR',
    'MODEL_CORRECT': 'MODEL_CORRECT',
    'REPRESENTATION_EQUIVALENT': 'REPRESENTATION_EQUIVALENT',
    'DATASET_AMBIGUITY': 'DATASET_AMBIGUITY',
    'INCONCLUSIVE': 'INCONCLUSIVE',
}

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def write_jsonl(path, rows):
    path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')

def extract_answer(value):
    matches = [match.group(1).strip() for match in (ANSWER_LINE.match(line) for line in str(value or '').splitlines()) if match]
    return matches[-1] if matches else None

def normalize_answer(value):
    text = re.sub(r'^\s*(?:final\s+answer|answer)\s*:\s*', '', str(value or '').lower().strip())
    text = re.sub(r'\s+', ' ', text)
    return text.strip(string.whitespace + '.,;:!?')

def exact_or_numeric_match(left, right):
    left, right = normalize_answer(left), normalize_answer(right)
    if left == right:
        return True
    try:
        return abs(float(left.replace(',', '').replace('%', '')) - float(right.replace(',', '').replace('%', ''))) <= 1e-6
    except ValueError:
        return False

def normalize_verdict(value):
    raw = str(value or '').upper().strip()
    raw = re.sub(r'[^A-Z]+', '_', raw).strip('_')
    return raw, VERDICT_ALIASES.get(raw, 'UNKNOWN')

raw_rows = read_jsonl(RAW_INPUT)
analysis_rows, resolved_rows, candidate_rows, review_rows = [], [], [], []

for row in raw_rows:
    label = row.get('teacher_json_parsed') or {}
    raw_verdict, verdict = normalize_verdict(label.get('verdict'))
    sft_final = row.get('extracted_final_answer') or extract_answer(row.get('sft_prediction_raw'))
    teacher_answer = label.get('correct_final_answer')
    corrected_response = label.get('corrected_response')
    corrected_final = extract_answer(corrected_response)
    flags = []
    if raw_verdict != verdict:
        flags.append(f'NORMALIZED_VERDICT:{raw_verdict}->{verdict}')
    sft_final_matches_gt = sft_final is not None and exact_or_numeric_match(sft_final, row['ground_truth'])
    teacher_answer_matches_gt = teacher_answer is not None and exact_or_numeric_match(teacher_answer, row['ground_truth'])
    corrected_final_matches_gt = corrected_final is not None and exact_or_numeric_match(corrected_final, row['ground_truth'])

    if row.get('teacher_status') != 'RAW_CAPTURED':
        route = 'CAPTURE_FAILURE_OR_UNPARSEABLE'
    elif sft_final_matches_gt:
        route = 'RESOLVED_MODEL_CORRECT'
        if verdict != 'MODEL_CORRECT':
            flags.append('SFT_FINAL_ALREADY_MATCHES_GT')
    elif verdict == 'REPRESENTATION_EQUIVALENT':
        route = 'RESOLVED_REPRESENTATION'
    elif verdict == 'MODEL_ERROR' and teacher_answer_matches_gt and corrected_final_matches_gt:
        route = 'DPO_CANDIDATE_PROVISIONAL'
    elif verdict == 'DATASET_AMBIGUITY':
        route = 'REVIEW_DATASET_AMBIGUITY'
    elif verdict == 'MODEL_ERROR':
        route = 'REVIEW_TEACHER_GT_CONFLICT'
        flags.append('MODEL_ERROR_NOT_GT_VALIDATED')
    elif verdict == 'INCONCLUSIVE':
        route = 'REVIEW_INCONCLUSIVE'
    else:
        route = 'REVIEW_UNKNOWN_VERDICT'

    analysis = {
        **row,
        'analysis_version': 'phase2c_teacher_analysis_v1',
        'teacher_verdict_raw': raw_verdict,
        'teacher_verdict_normalized': verdict,
        'sft_final_answer_rechecked': sft_final,
        'teacher_corrected_final_answer': corrected_final,
        'sft_final_matches_gt': sft_final_matches_gt,
        'teacher_answer_matches_gt': teacher_answer_matches_gt,
        'corrected_final_matches_gt': corrected_final_matches_gt,
        'analysis_route': route,
        'analysis_flags': flags,
        'is_dpo_pair': route == 'DPO_CANDIDATE_PROVISIONAL',
    }
    analysis_rows.append(analysis)
    if route.startswith('RESOLVED_'):
        resolved_rows.append(analysis)
    elif route == 'DPO_CANDIDATE_PROVISIONAL':
        candidate_rows.append({
            'pair_version': 'phase2c_dpo_candidate_v1',
            'dataset_index': analysis['dataset_index'],
            'image_dataset': analysis['image_dataset'],
            'image_split': analysis['image_split'],
            'image_index': analysis['image_index'],
            'prompt': analysis['question'],
            'chosen': corrected_response,
            'rejected': analysis['sft_prediction_raw'],
            'ground_truth': analysis['ground_truth'],
            'teacher_verdict_normalized': verdict,
            'analysis_flags': flags,
            'manual_audit_status': 'PENDING',
        })
    else:
        review_rows.append(analysis)

all_path = RESULTS_DIR / f'{OUTPUT_PREFIX}_analysis_all.jsonl'
resolved_path = RESULTS_DIR / f'{OUTPUT_PREFIX}_resolved_correct.jsonl'
candidates_path = RESULTS_DIR / f'{OUTPUT_PREFIX}_dpo_candidates_provisional.jsonl'
review_path = RESULTS_DIR / f'{OUTPUT_PREFIX}_review.jsonl'
report_path = RESULTS_DIR / f'{OUTPUT_PREFIX}_analysis_report.json'
write_jsonl(all_path, analysis_rows)
write_jsonl(resolved_path, resolved_rows)
write_jsonl(candidates_path, candidate_rows)
write_jsonl(review_path, review_rows)

report = {
    'raw_input': str(RAW_INPUT),
    'records': len(analysis_rows),
    'routes': dict(Counter(row['analysis_route'] for row in analysis_rows)),
    'teacher_verdicts_normalized': dict(Counter(row['teacher_verdict_normalized'] for row in analysis_rows)),
    'provisional_dpo_candidates': len(candidate_rows),
    'manual_audit_required_before_dpo': True,
    'outputs': {
        'all': str(all_path), 'resolved': str(resolved_path),
        'dpo_candidates_provisional': str(candidates_path), 'review': str(review_path),
    },
}
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))


## 3. Inspect provisional DPO pairs

Verify each `chosen` answer and reasoning against the chart before changing `manual_audit_status` to approved. These records are intentionally not yet a train-ready DPO dataset.


In [ ]:
candidates = read_jsonl(candidates_path)
print('Provisional DPO candidates:', len(candidates))
for row in candidates[:5]:
    print('\nINDEX:', row['dataset_index'])
    print('QUESTION:', row['prompt'])
    print('CHOSEN:\n' + row['chosen'])
    print('REJECTED:\n' + row['rejected'])
    print('GT:', row['ground_truth'])
